In [ ]:
import json
import os
import shutil

import yaml
from dotenv import load_dotenv
from IPython.display import Image
from roboflow import Roboflow
from ultralytics import YOLO

os.chdir("..")
load_dotenv(os.path.join("config", ".env"))

#### Setup directory

In [ ]:
pth_settings = "/Users/user/Library/Application Support/Ultralytics/settings.yaml"

with open(pth_settings, "r") as file:
    data = yaml.safe_load(file)

data["datasets_dir"] = os.getcwd()

with open(pth_settings, "w") as file:
    yaml.dump(data, file)

In [ ]:
pth_settings = "/Users/user/Library/Application Support/Ultralytics/settings.json"

with open(pth_settings, "r") as file:
    data = json.load(file)

data["datasets_dir"] = os.getcwd()

with open(pth_settings, "w") as file:
    json.dump(data, file)

#### Download dataset

In [ ]:
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("jack-chan-edpdi").project("supermarketscanner")
dataset = project.version(5).download("yolov8")

os.makedirs("datasets", exist_ok=True)
shutil.move("SupermarketScanner-5", "datasets")

#### Train model

In [ ]:
model = YOLO(os.path.join("models", "yolov8n-seg.pt"))

_ = model.train(
    data=os.path.join("datasets", "SupermarketScanner-5", "data.yaml"),
    epochs=64,
    name="smkt_scanner",
)

In [ ]:
Image(os.path.join("runs", "segment", "smkt_scanner", "results.png"))

In [ ]:
Image(os.path.join("runs", "segment", "smkt_scanner", "confusion_matrix.png"))

In [ ]:
model.val(
    split="test",
    name="smkt_scanner_val",
)

#### Centralise model

In [ ]:
shutil.copy2(
    os.path.join("runs", "segment", "smkt_scanner", "weights", "best.pt"),
    os.path.join("models", "smkt_scanner.pt"),
)